# Compute the SNR of pediatric data

## Goals:
1. Import the preprocessed data from npz and json files
2. Calculate the Signal-to-Noise Ratio (SNR) for each epoch (for use in stats)
3. Export the SNR


# Import Libraries

In [1]:
# Standard libraries
import json
import numpy as np
import pandas as pd
import scipy.signal as signal

# Custom libraries
from Functions import processing

# Batch Import Epoched Data and Settings

In [2]:
# Load list of files to import
files = [  
    "sub-P001_ses-S001_task-T1_run-001_eeg","sub-P002_ses-S001_task-T1_run-001_eeg","sub-P003_ses-S001_task-T1_run-001_eeg","sub-P004_ses-S001_task-T1_run-001_eeg",
    "sub-P005_ses-S001_task-T1_run-001_eeg","sub-P006_ses-S001_task-T1_run-001_eeg","sub-P007_ses-S001_task-T1_run-001_eeg","sub-P008_ses-S001_task-T1_run-001_eeg", 
    "sub-P009_ses-S001_task-T1_run-001_eeg","sub-P010_ses-S001_task-T1_run-001_eeg","sub-P011_ses-S001_task-T1_run-001_eeg","sub-P012_ses-S001_task-T1_run-001_eeg",
    "sub-P013_ses-S001_task-T1_run-001_eeg","sub-P014_ses-S001_task-T1_run-001_eeg","sub-P015_ses-S001_task-T1_run-001_eeg","sub-P016_ses-S001_task-T1_run-001_eeg",
    "sub-P017_ses-S001_task-T1_run-001_eeg","sub-P018_ses-S001_task-T1_run-001_eeg","sub-P019_ses-S001_task-T1_run-001_eeg","sub-P020_ses-S001_task-T1_run-001_eeg",
    "sub-P021_ses-S001_task-T1_run-001_eeg","sub-P022_ses-S001_task-T1_run-001_eeg","sub-P023_ses-S001_task-T1_run-001_eeg","sub-P024_ses-S001_task-T1_run-001_eeg",
    "sub-P025_ses-S001_task-T1_run-001_eeg","sub-P026_ses-S001_task-T1_run-001_eeg","sub-P027_ses-S001_task-T1_run-001_eeg","sub-P028_ses-S001_task-T1_run-001_eeg",
    "sub-P029_ses-S001_task-T1_run-001_eeg","sub-P030_ses-S001_task-T1_run-001_eeg","sub-P031_ses-S001_task-T1_run-001_eeg","sub-P032_ses-S001_task-T1_run-001_eeg",
    "sub-P033_ses-S001_task-T1_run-001_eeg","sub-P034_ses-S001_task-T1_run-001_eeg","sub-P035_ses-S001_task-T1_run-001_eeg","sub-P036_ses-S001_task-T1_run-001_eeg",
    "sub-P037_ses-S001_task-T1_run-001_eeg","sub-P038_ses-S001_task-T1_run-001_eeg","sub-P039_ses-S001_task-T1_run-001_eeg","sub-P040_ses-S001_task-T1_run-001_eeg",
    "sub-P041_ses-S001_task-T1_run-001_eeg","sub-P042_ses-S001_task-T1_run-001_eeg","sub-P043_ses-S001_task-T1_run-001_eeg","sub-P044_ses-S001_task-T1_run-001_eeg",
    "sub-P045_ses-S001_task-T1_run-001_eeg","sub-P046_ses-S001_task-T1_run-001_eeg","sub-P047_ses-S001_task-T1_run-001_eeg"
]

# Get unique subject IDs
subject_ids = [file.split('_')[0] for file in files]
unique_subject_ids = list(set(subject_ids))

# Preallocate variables to store EEG data and settings
eeg_epochs = [None] * len(files)
settings = [None] * len(files)

# Import data
for f, file in enumerate(files):
    for sub in subject_ids:
        if sub == file.split('_')[0]:
            # Import EEG data, since it is stored in a compressed numpy file (.npz) we need to use the np.load function 
            loaded_data = np.load(f"Data\\PedsData\\EEG\\{sub}\\ses-S001\\eeg\\{file}.npz", allow_pickle=True)

            # Access the data for each stimulus
            eeg_epochs[f] = {stim_label: loaded_data[stim_label] for stim_label in loaded_data.files}

            # Import settings
            with open(f"Data\\PedsData\\EEG\\{sub}\\ses-S001\\eeg\\{file}.json", "r") as file_object: settings[f] = json.load(file_object)

# Calculate PSD of all Epochs

In [3]:
# PSD settings
window_size = 5 

# Preallocate variables
eeg_f = [None] * len(files)
eeg_pxx = [None] * len(files)

# Compute PSD for each file
for f, file in enumerate(files):
    eeg_f[f] = {}
    eeg_pxx[f] = {}

    # Compute PSD for each stimulus
    for stim_label, epochs in eeg_epochs[f].items():
        eeg_f[f][stim_label] = []
        eeg_pxx[f][stim_label] = []

        # Compute PSD for each epoch
        for epoch in epochs:
            f_values, pxx_values = signal.welch(
                x=epoch,
                fs=settings[f]["eeg_srate"],
                nfft=int(window_size * settings[f]["eeg_srate"]),
                nperseg=window_size * settings[f]["eeg_srate"],
                noverlap= (window_size * settings[f]["eeg_srate"]) * 0.5,  # 50% overlap between windows
            )
            eeg_f[f][stim_label].append(f_values)
            eeg_pxx[f][stim_label].append(pxx_values)

        # Convert lists to arrays for consistency
        eeg_f[f][stim_label] = np.array(eeg_f[f][stim_label])
        eeg_pxx[f][stim_label] = np.array(eeg_pxx[f][stim_label])

c:\Users\admin\miniconda3\envs\bessy\Lib\site-packages\scipy\signal\_spectral_py.py:790: UserWarning: nperseg = 1280 is greater than input length  = 1279, using nperseg = 1279
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
c:\Users\admin\miniconda3\envs\bessy\Lib\site-packages\scipy\signal\_spectral_py.py:790: UserWarning: nperseg = 1280 is greater than input length  = 1278, using nperseg = 1278
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,


# Compute SNR for all Epochs
- SNR is calculated for each epoch and then averaged per stimulus

In [4]:
# Settings
noise_band = 1    # Single-sided noise band [Hz]
nharms = 2        # Number of harmonics used
stim_freq = 10    # Stimulus frequency in Hz
db_out = True     # Boolean to get output in dB

# Collect all unique channel names from all files and extract only occipital channels
all_channel_names = set()
for s in settings:
    all_channel_names.update(s["new_ch_names"])
all_channel_names = sorted(list(all_channel_names))  # Keep consistent order
oz_channels = [ch for ch in all_channel_names if ch in ['O1', 'Oz', 'O2']]

# Initialize containers - store one value per stimulus per participant
snr = [None] * len(files)

# Compute SNR with per-file channel alignment
for f0 in range(len(files)):
    stim_labels = list(settings[f0]["stimuli"].values())
    temp_snr = np.zeros(len(stim_labels))  # shape: (n_stimuli,)
    
    # Get channel indices for this file
    file_channels = settings[f0]["new_ch_names"]
    ch_idx_map = {ch: i for i, ch in enumerate(file_channels)}
    
    # Find which occipital channels are available in this file
    available_oz_channels = [ch for ch in oz_channels if ch in ch_idx_map]
    
    for stim_idx, stim_label in settings[f0]["stimuli"].items():
        stim_num = stim_idx 
        s = stim_labels.index(stim_label)
        
        channel_snr_list = []
        num_epochs = eeg_pxx[f0][stim_label].shape[0]
        
        for epoch in range(num_epochs):
            snr_epoch = processing.ssvep_snr(
                f=eeg_f[f0][stim_label][epoch],  # shape: (n_freqs,)
                pxx=eeg_pxx[f0][stim_label][epoch, :, :],  # shape: (n_channels, n_freqs)
                stim_freq=stim_freq,
                noise_band=noise_band,
                nharms=nharms,
                db_out=db_out
            )
            channel_snr_list.append(snr_epoch)  # shape: (n_channels,)
        
        # Average across epochs → (n_channels,)
        mean_snr = np.mean(np.stack(channel_snr_list), axis=0)
        
        # Average across available occipital channels → 1 value
        if available_oz_channels:
            occipital_snr_values = [mean_snr[ch_idx_map[ch]] for ch in available_oz_channels]
            mean_occipital_snr = np.mean(occipital_snr_values)
        else:
            mean_occipital_snr = np.nan  # No occipital channels available
        
        # Store the single SNR value for this stimulus
        temp_snr[s] = mean_occipital_snr
    
    snr[f0] = temp_snr

# Export SNR

In [5]:
save_snr = True

# Preallocate empty list to store all dataframes
dfs = []

for f0, file in enumerate(files):
    # Get participant ID from filename (adjust as needed)
    participant_id = file.split('/')[-1].split('_')[0]
    
    # Get stimulus labels for this file
    stim_labels = list(settings[f0]["stimuli"].values())
    
    # Create a row for this participant
    data_dict = {}
    
    # Add each stimulus SNR value
    for stim_idx, stim_label in settings[f0]["stimuli"].items():
        s = stim_labels.index(stim_label)
        data_dict[stim_label] = snr[f0][s]
    
    # Create DataFrame for this participant
    df_participant = pd.DataFrame([data_dict])
    
    # Add participant info as a column
    df_participant.insert(0, 'Participant', participant_id)
    
    dfs.append(df_participant)

# Concatenate all DataFrames
if dfs:
    snr_df = pd.concat(dfs, ignore_index=True)
    
    # Save SNRs to CSV
    if save_snr:
        snr_df.to_csv("Data\\PedsData\\EEG\\snr.csv", index=False)
        print(f"Saved SNR data for {len(snr_df)} participants to CSV")
else:
    print("No data to save!")

Saved SNR data for 47 participants to CSV
